In [ ]:
import json
import os

import numpy as np
import pandas as pd
from dotenv import load_dotenv
from rich.pretty import pprint
from sklearn.manifold import TSNE
from tensorflow.keras.models import load_model

from open_ai_embedder import OpenAiEmbedder
from visualizer import Visualizer

load_dotenv()

In [ ]:
# initialize embedder
embedder = OpenAiEmbedder(api_key=os.getenv("OPENAI_API_KEY"))

In [ ]:
# load and inspect animals facts
with open("labeld_animal_facts.json", "r") as f:
    animal_facts = json.load(f)
animal_facts_df = pd.DataFrame(animal_facts)
animal_facts_df.head(3)

In [ ]:
# show unique values
print(animal_facts_df["animal"].unique())
print(animal_facts_df["weightClass"].unique())
print(animal_facts_df["trophicLevel"].unique())

In [ ]:
texts = animal_facts_df["text"].to_list()


def load_embeddings(should_recreate_embeddings=False):
    path = "labeled_animal_facts_with_embeddings.json"
    # if file exists, load it
    if os.path.exists(path) and not should_recreate_embeddings:
        with open(path, "r") as f:
            facts_with_embeddings = json.load(f)
            text_embeddings = [fact["embedding"] for fact in facts_with_embeddings]
    else:
        text_embeddings = embedder.batch_embed(texts)
    return text_embeddings


text_embeddings = load_embeddings(should_recreate_embeddings=False)

len(text_embeddings), len(animal_facts_df)

In [ ]:
# add embeddings to df
animal_facts_df["embedding"] = text_embeddings
animal_facts_df.keys()

In [ ]:
# reduce dimensions for 2d plot

def reduce_dimensions(embeddings, dimensions=2):
    tsne = TSNE(n_components=dimensions, random_state=42, init="pca")
    reduced_embeddings = tsne.fit_transform(embeddings)
    return reduced_embeddings


reduced_embeddings_2d = reduce_dimensions(embeddings=np.array(text_embeddings), dimensions=2)
df_with_2d_embeddings = pd.DataFrame({
    "text": texts,
    "x": reduced_embeddings_2d[:, 0],
    "y": reduced_embeddings_2d[:, 1],
    "animal": animal_facts_df["animal"],
    "weightClass": animal_facts_df["weightClass"],
    "trophicLevel": animal_facts_df["trophicLevel"],
})


In [ ]:
Visualizer.scatter(df_with_2d_embeddings, hover_data_key="text", color_key="animal")

In [ ]:
Visualizer.scatter(df_with_2d_embeddings, hover_data_key="text", color_key="weightClass")

In [ ]:
Visualizer.scatter(df_with_2d_embeddings, hover_data_key="text", color_key="trophicLevel")

In [ ]:
sample_embedding = animal_facts_df.head(1)["embedding"].to_list()[0]
type(sample_embedding), len(sample_embedding)

In [ ]:
from animal_fact_model_trainer import AnimalFactModelTrainer

model_trainer = AnimalFactModelTrainer(animal_facts_df)

In [ ]:
model_animal, history_animal, mapping_animal = model_trainer.train_model(
    target_column="animal",
    model_name_prefix="animal_model",
    epochs=20,
    batch_size=16,
    patience=3
)

In [ ]:
model_weight, history_weight, mapping_weight = model_trainer.train_model(
    target_column="weightClass",
    model_name_prefix="animal_model",
    epochs=20,
    batch_size=16,
    patience=3
)

In [ ]:
model_trophic, history_trophic, mapping_trophic = model_trainer.train_model(
    target_column="trophicLevel",
    model_name_prefix="animal_model",
    epochs=10,
    batch_size=8,
    patience=3
)

In [ ]:
def classify_text(text, model_prefix="animal_model", n_predictions=1, embedder=embedder):
    embedding = embedder.embed(text)
    embedding_array = np.array(embedding, dtype=np.float32).reshape(1, -1)  # shape=(1,1536)

    predictions = {}
    categories = ["animal", "weightClass", "trophicLevel"]

    for category in categories:
        model_file = f"{model_prefix}_{category}.keras"
        mapping_file = f"{model_prefix}_{category}_mapping.json"
        model = load_model(model_file, compile=False)

        # corresponding mapping
        with open(mapping_file, "r") as f:
            index_to_label = json.load(f)

        pred_probs = model.predict(embedding_array, verbose=0)[0]
        top_n_indices = np.argsort(pred_probs)[::-1][:n_predictions]
        top_preds = []
        for idx in top_n_indices:
            label = index_to_label.get(str(idx), index_to_label.get(idx))
            top_preds.append((label, float(pred_probs[idx])))
        predictions[category] = top_preds
    pprint((text, predictions))

In [ ]:
#lets start with something easy:
classify_text("a cat is a small predatory mammal", n_predictions=1)
#classify_text("a bee collects pollen from flowers", n_predictions=3)
#classify_text("parrots are known for their colorful feathers", n_predictions=1)
#classify_text("the black widow spider is a venomous arachnid", n_predictions=1)


In [ ]:
#medium examples (real life examples)
#classify_text("Lizards are the most common reptiles in Bangkok, and are a big part of natural pest control.", n_predictions=1)

#classify_text("Geckos have soft skin, large eyes, and famously run up and down the city’s walls. Geckos are usually seen at night, but can be found during the day inside darker areas. The following geckos are found in Bangkok: Flat-tailed House Gecko, Stump-toed Gecko, Spiny-tailed House Gecko", n_predictions=1)


#classify_text("The Malayan peacock-pheasant is vulnerable to extinction due to deforestation.", n_predictions=1)

#classify_text("Praying mantises are closely related to termites and cockroaches. The word mantis is Greek and means ‘prophet’ or ‘fortune teller’. It is more or less well­known that female mantises eat the male after reproduction", n_predictions=1)


#classify_text("There are over 2500 species of cicada in the world. Cicadas have large compound eyes on each side of the head; they also have three very small glistening simple eyes on the top of the head, and transparent well­veined wings", n_predictions=3)


In [ ]:
#harder examples
#classify_text("bats find their prey in the dark with their echolocation", n_predictions=1)
#classify_text("polar bears are known for their thick fur", n_predictions=2)
#classify_text("it is a proven fact that cats are cuter than dogs", n_predictions=2)
#classify_text("dumbo the elephant can fly in the clouds", n_predictions=2)
#classify_text("winnie the pooh loves to play with his friends and eat honey", n_predictions=2)
#classify_text("chang beer is popular in thailand ", n_predictions=3)
#classify_text("a meowing thing that has 4 paws and is super cute", n_predictions=3)
#classify_text("trex is the coolest dino", n_predictions=3)